In [ ]:
# ============================================================
# 0. 基础库
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, KernelPCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import Isomap, TSNE

import warnings
warnings.filterwarnings('ignore')

try:
    import umap
    HAS_UMAP = True
except:
    HAS_UMAP = False

# ============================================================
# 1. 全局设置
# ============================================================
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

COLORS = ['#E41A1C', '#377EB8', '#4DAF4A', '#984EA3', '#FF7F00',
          '#FFFF33', '#A65628', '#F781BF', '#66C2A5', '#FC8D62']

# ============================================================
# 2. 数据
# ============================================================
digits = load_digits()
X = digits.data
y = digits.target

print("样本数:", X.shape[0])
print("特征数:", X.shape[1])

# ============================================================
# 3. 划分 + 标准化
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# ============================================================
# PCA
# ============================================================
print("\n[PCA]")

pca = PCA()
X_pca = pca.fit_transform(X_train_scaled)

ev = np.cumsum(pca.explained_variance_ratio_)
n95 = np.argmax(ev >= 0.95) + 1

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
for i in range(10):
    plt.scatter(X_pca[y_train==i,0],
                X_pca[y_train==i,1],
                s=6, alpha=0.6, c=COLORS[i])
plt.title("PCA")

plt.subplot(1,2,2)
plt.plot(ev)
plt.axhline(0.95, color='r')
plt.axvline(n95, color='r')
plt.title("PCA累计贡献率")

plt.tight_layout()
plt.savefig("pca.png", dpi=150)
plt.close()

# ============================================================
# LDA（规范版）
# ============================================================
print("\n[LDA]")

lda = LinearDiscriminantAnalysis()

X_lda = lda.fit_transform(X_train_scaled, y_train)

cv = cross_val_score(lda, X_train_scaled, y_train, cv=5)

plt.figure(figsize=(6,5))
for i in range(10):
    plt.scatter(X_lda[y_train==i,0],
                X_lda[y_train==i,1],
                s=6, alpha=0.6, c=COLORS[i])

plt.title(f"LDA | CV={cv.mean():.3f}")
plt.xlabel("LD1")
plt.ylabel("LD2")

plt.tight_layout()
plt.savefig("lda.png", dpi=150)
plt.close()

# ============================================================
# KPCA
# ============================================================
print("\n[KPCA]")

gammas = [0.001, 0.01, 0.1]

fig, axes = plt.subplots(1,3, figsize=(15,4))

for ax, g in zip(axes, gammas):
    kpca = KernelPCA(n_components=2, kernel='rbf', gamma=g)
    X_k = kpca.fit_transform(X_train_scaled)

    for i in range(10):
        ax.scatter(X_k[y_train==i,0],
                   X_k[y_train==i,1],
                   s=5, alpha=0.6, c=COLORS[i])

    ax.set_title(f"KPCA γ={g}")

plt.tight_layout()
plt.savefig("kpca.png", dpi=150)
plt.close()

# ============================================================
# Isomap（多参数）
# ============================================================
print("\n[Isomap]")

neighbors_list = [5, 15, 30]

fig, axes = plt.subplots(1,3, figsize=(15,4))

for ax, k in zip(axes, neighbors_list):
    iso = Isomap(n_components=2, n_neighbors=k)
    X_i = iso.fit_transform(X_train_scaled)

    for i in range(10):
        ax.scatter(X_i[y_train==i,0],
                   X_i[y_train==i,1],
                   s=5, alpha=0.6, c=COLORS[i])

    ax.set_title(f"Isomap k={k}")

plt.tight_layout()
plt.savefig("isomap.png", dpi=150)
plt.close()

# ============================================================
# t-SNE（PCA预处理）
# ============================================================
print("\n[t-SNE]")

X_pre = PCA(n_components=50).fit_transform(X_train_scaled)

fig, axes = plt.subplots(1,2, figsize=(12,5))

for ax, p in zip(axes, [30, 50]):
    tsne = TSNE(
        n_components=2,
        perplexity=p,
        max_iter=1000,
        random_state=42
    )

    X_t = tsne.fit_transform(X_pre)

    for i in range(10):
        ax.scatter(X_t[y_train==i,0],
                   X_t[y_train==i,1],
                   s=5, alpha=0.6, c=COLORS[i])

    ax.set_title(f"t-SNE perplexity={p}")

plt.tight_layout()
plt.savefig("tsne.png", dpi=150)
plt.close()

# ============================================================
# UMAP（PCA预处理）
# ============================================================
if HAS_UMAP:
    print("\n[UMAP]")

    X_pre_umap = PCA(n_components=50).fit_transform(X_train_scaled)

    fig, axes = plt.subplots(1,2, figsize=(12,5))

    for ax, nn in zip(axes, [15, 30]):
        reducer = umap.UMAP(
            n_components=2,
            n_neighbors=nn,
            min_dist=0.1,
            random_state=42
        )

        X_u = reducer.fit_transform(X_pre_umap)

        for i in range(10):
            ax.scatter(X_u[y_train==i,0],
                       X_u[y_train==i,1],
                       s=5, alpha=0.6, c=COLORS[i])

        ax.set_title(f"UMAP n_neighbors={nn}")

    plt.tight_layout()
    plt.savefig("umap.png", dpi=150)
    plt.close()

# ============================================================
print("\n全部完成 ✔")